# Decision Tree (Pipeline B)

**Model**: `DecisionTreeRegressor` / **Target**: gdpc1  
**Variables**: Cat3 (53 + COVID = 56) / **n_lags**: 4  
**Scaling**: NO / **HP tuning**: YES / **10-model averaging**: YES

In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from scipy.stats import norm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]
sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import (gen_lagged_data, flatten_data, mean_fill_dataset,
                     get_features, load_data, PREDICTIONS_DIR, TARGET)
SEED = 42; np.random.seed(SEED)
TRAIN_START = '1959-01-01'; TRAIN_END = '2007-12-31'; VAL_END = '2016-12-31'
TEST_START = '2017-01-01'; TEST_END = '2025-12-31'
N_LAGS = 4; N_MODELS = 10; VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0}
print('DT — Cat3, n_lags=4')

DT — Cat3, n_lags=4


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
# Phase A: HP tuning
tune = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_f = mean_fill_dataset(tune, tune)
tune_fl = flatten_data(tune_f, TARGET, N_LAGS)
tune_fl = tune_fl.loc[tune_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)
fcols = [c for c in tune_fl.columns if c != 'date' and c != TARGET and any(c == f or c.startswith(f + '_') for f in features)]
tscv = TimeSeriesSplit(n_splits=5)
dt_grid = {'max_depth': [None, 5, 10, 20], 'min_samples_leaf': [1, 3, 5]}
gs = GridSearchCV(DecisionTreeRegressor(criterion='squared_error', random_state=SEED),
    dt_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=1)
gs.fit(tune_fl[fcols].values, tune_fl[TARGET].values)
bp = gs.best_params_
print(f'Best DT params: {bp}')

Best DT params: {'max_depth': None, 'min_samples_leaf': 5}


In [3]:
test_quarters = monthly[(monthly['date'] >= TEST_START) & (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])]['date'].tolist()
predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0
for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])
    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= fc)].reset_index(drop=True)
        train_m = gen_lagged_data(metadata, train.copy(), fc, lag=0)
        train_f = mean_fill_dataset(train_m, train_m)
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)
        if len(train_fl) < 10:
            predictions[vn].append(np.nan); continue
        try:
            models = []
            for k in range(N_MODELS):
                m = DecisionTreeRegressor(criterion='squared_error', max_depth=bp['max_depth'],
                    min_samples_leaf=bp['min_samples_leaf'], random_state=SEED+k)
                m.fit(train_fl[fcols].values, train_fl[TARGET].values)
                models.append(m)
            tr = monthly[monthly['date'] == fc].reset_index(drop=True)
            tr_m = gen_lagged_data(metadata, tr.copy(), fc, lag=0)
            tr_f = mean_fill_dataset(train_m, tr_m)
            ctx = train_f.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb = pd.concat([ctx, tr_f], ignore_index=True)
            cmb_fl = flatten_data(cmb, TARGET, N_LAGS)
            tr_fl = cmb_fl.tail(1)
            preds = [m.predict(tr_fl[fcols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1
    if (i+1) % 8 == 0:
        print(f'  {i+1}/{len(test_quarters)}')
print(f'Done. {failed} failures.')

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR, f'dt_{vn}.csv'), index=False)
    print(f'Saved dt_{vn}.csv')

Saved dt_m1.csv
Saved dt_m2.csv
Saved dt_m3.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum() > 0 else np.nan
def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum() > 0 else np.nan

panels = {
    'Pre-COVID  (2017-2019)': ('2017-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2017-2025)': ('2017-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print(f'DT best params: {bp}')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print(f'--- {pn} ---')
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print(f'  {vn}  RMSFE={rmse(a[m], pp[m]):.6f}  MAE={mae(a[m], pp[m]):.6f}')

DT best params: {'max_depth': None, 'min_samples_leaf': 5}
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002429  MAE=0.001813
  m2  RMSFE=0.002824  MAE=0.002171
  m3  RMSFE=0.002674  MAE=0.002155
--- COVID      (2020-2021) ---
  m1  RMSFE=0.042938  MAE=0.026910
  m2  RMSFE=0.042671  MAE=0.026649
  m3  RMSFE=0.042814  MAE=0.026693
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004392  MAE=0.003359
  m2  RMSFE=0.004552  MAE=0.003440
  m3  RMSFE=0.004520  MAE=0.003409
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019512  MAE=0.007899
  m2  RMSFE=0.019432  MAE=0.008004
  m3  RMSFE=0.019494  MAE=0.008004
